In [ ]:
## created a RAG model with chat history along with citations (details like doc name, page no.)

## for this we store chat history in a list which gets append with each conversation
## then the current question and chat history get passed through the LLM to rewrite the question (which also understand the context)
## then that rewritten question get pass throght the retrival pipeline to answer it correctly

## we are also adding the citation details along with answer like document name, page number
## its getting done by adding the metadata along withe page content and by modifying the system prompt

In [1]:
import tqdm
import os
import shutil
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

/Users/ashish/interview prep/langchain codes/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ashish/interview prep/langchain codes/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- 1. SETUP MODELS & DATA ---
os.environ["GOOGLE_API_KEY"] = "put your API key over here"
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11490.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# --- 2. LOAD & CHUNK PDF ---
if os.path.exists("./gemini_rag_db"):
    shutil.rmtree("./gemini_rag_db")

In [5]:
pdf_path = "MTF_Activation.pdf" # Replace with your PDF filename
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# docs

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

print(type(chunks))
print(len(chunks))
print(chunks[1].page_content)

<class 'list'>
19
manner or by physical mode.
3) KSL at all times shall have the liberty to exercise its right in its 
sole discretion to determine the extent to which the MTF to be 
made to the Client.
4) Pay interest at the rate agreed under the voluntary terms and 
conditions at the time of opening the client’s account and/or 
modified and communicated from time to time by KSL.
5) If the transaction is entered under MTF , there will not be any 
further confirmation that it is margin trading transaction other 
than contract note.
6) Transaction/s to be considered for exposure to MTF shall be 
informed to KSL in writing or in any other irrefutable mode of 
communication not later than T+1 day, else the same shall be 
considered under normal trading facility. Additional exposure 
over debit balance (arising out of trade executed under normal 
trading facility), beyond fifth trading day reckoned  from pay-
in date, may be granted under MTF to the extent the Client is


In [6]:
from tqdm import tqdm
vector_db = Chroma.from_documents(
    documents=[doc for doc in tqdm(chunks, desc="Processing Chunks")], 
    embedding=embeddings, 
    persist_directory="./gemini_rag_db"
)

Processing Chunks: 100%|██████████| 19/19 [00:00<00:00, 240760.65it/s]


In [7]:
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

In [8]:
# --- 3. THE "RE-WRITER" CHAIN (Call #1) ---
# This turns history + new question into a standalone search query
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# This sub-chain creates the search query
history_aware_retriever = (
    contextualize_q_prompt 
    | llm 
    | StrOutputParser() 
    | retriever
)

In [9]:
# --- 4. THE "ANSWER" CHAIN (Call #2) ---
# system_prompt = (
#     "You are an expert research assistant. "
#     "Use the following pieces of retrieved context to answer the question. "
#     "If you don't know the answer, say that you don't know. "
#     "\n\n"
#     "{context}"
# )

system_prompt = (
    "You are an expert research assistant. "
    "Use the following pieces of retrieved context to answer the question. "
    "Every time you provide information, you must cite the document name and page number "
    "provided in the context (e.g., [Source: filename.pdf, Page: 5]). "
    "If you don't know the answer, say that you don't know. "
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)

def format_docs_with_id(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "Unknown")
        content = f"CONTENT: {doc.page_content}\nSOURCE: {source}\nPAGE: {page}"
        formatted.append(content)
    return "\n\n---\n\n".join(formatted)

# The final assembly line
rag_chain = (
    RunnablePassthrough.assign(
        context=history_aware_retriever | format_docs_with_id
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)


In [10]:
# --- 5. CHAT LOOP WITH MEMORY MANAGEMENT - only consider last 3 Q&A pairs ---
chat_history = [] # This stores our memory

print("--- AI Research Assistant Ready! (Type 'exit' to stop) ---")

while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    # Execute the chain
    response = rag_chain.invoke({
        "input": user_input,
        "chat_history": chat_history[-6:] 
        # it'll only keep last 6 chats (3 Q&A pairs) to keep the context recent relevant and 
        # also to avoid memory issues and hullucinations of the model
    })

    print(f"\nAI: {response}")

    # Update History: Add both the question and the answer to the list
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))

--- AI Research Assistant Ready! (Type 'exit' to stop) ---

AI: The document is primarily about the **Margin Trading Facility (MTF) Activation** [Source: MTF_Activation.pdf, Page: 0]. It outlines the terms and conditions for availing this facility, including the rights and obligations of the client and KSL (the provider) [Source: MTF_Activation.pdf, Page: 0]. It also details various circumstances under which the MTF facility may be withdrawn, such as the death or disability of the client, default in payments, or other conditions deemed prejudicial to KSL's interests [Source: MTF_Activation.pdf, Page: 1].
